# 03 · PCA and clustering
Model logic lives in `src/model.py`. This notebook reads `features.parquet` for diagnostics and `clusters.parquet` / `metrics.json` for the fitted result.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
import json
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from src.data_io import read_features, read_clusters
from src import model

In [ ]:
feats = read_features()
X_scaled = StandardScaler().fit_transform(feats[config.FEATURES])
pca = PCA().fit(X_scaled)
ev = pd.DataFrame({"explained": pca.explained_variance_ratio_,
                   "cumulative": pca.explained_variance_ratio_.cumsum()},
                  index=[f"PC{i+1}" for i in range(len(config.FEATURES))])
ev

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ev.index, ev["explained"], marker="o"); ax[0].set_title("Scree")
ax[1].plot(ev.index, ev["cumulative"], marker="o"); ax[1].axhline(.8, ls="--"); ax[1].axhline(.9, ls="--")
ax[1].set_title("Cumulative variance"); plt.show()

In [ ]:
loadings = pd.DataFrame(pca.components_.T, index=config.FEATURES,
                        columns=[f"PC{i+1}" for i in range(len(config.FEATURES))])
loadings.style.background_gradient(cmap="coolwarm", axis=None)

## Choosing k

In [ ]:
sweep = model.k_sweep(feats, ks=range(2, 11))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(sweep.k, sweep.inertia, marker="o"); ax[0].set_title("Elbow (inertia)")
ax[1].plot(sweep.k, sweep.silhouette, marker="o"); ax[1].set_title("Silhouette")
plt.show(); sweep

## Fitted model

In [ ]:
clusters = read_clusters()
metrics = json.loads(config.metrics_path().read_text())
print({k: metrics[k] for k in ["n_rows", "cumulative_variance_kept", "silhouette", "cluster_sizes"]})

In [ ]:
pcs = [f"PC{i+1}" for i in range(config.N_COMPONENTS)]
sns.pairplot(clusters[pcs + ["cluster"]], hue="cluster", palette="tab10", corner=True,
             plot_kws={"alpha": 0.6, "s": 12})
plt.suptitle("PCA space by cluster", y=1.02); plt.show()

## Cluster profiles
Cluster ids are arbitrary and can change after any upstream edit. Name archetypes from these profiles each run (original Colab run: Low-Commission Mature, Inactive, High-Commission Mature, Elite Performance, Emerging Low-Commission, Volatile).

In [ ]:
med = model.cluster_profiles(clusters, "median")
display(med.round(3))
display(model.cluster_profiles(clusters, "mean").round(3))

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(model.profile_zscores(med), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Cluster feature profiles (z-scored medians)"); plt.show()